In [0]:

from pyspark.sql.functions import col, explode

# Step 1: Read the original JSON source
file_path = "/Volumes/reconciliation/default/reconciliation_files/retail_source.json"

source_df = spark.read.option("multiLine", True).json(file_path)

# Step 2: Read the actual Delta target
target_df = spark.table("reconciliation.default.retail_orders")

print("Source loaded successfully")
print("Target loaded successfully")


In [0]:

expected_df = source_df.select(
    col("customer_id"),
    col("customer_name"),
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    explode(col("orders")).alias("order")
).select(
    col("customer_id"),
    col("customer_name"),
    col("city"),
    col("state"),
    col("order.order_id").alias("order_id"),
    col("order.order_date").alias("order_date"),
    explode(col("order.items")).alias("item")
).select(
    col("customer_id"),
    col("customer_name"),
    col("city"),
    col("state"),
    col("order_id"),
    col("order_date"),
    col("item.product_id").alias("product_id"),
    col("item.quantity").alias("quantity"),
    col("item.amount").alias("amount")
)

display(expected_df)


In [0]:
validation_results = []

def add_validation_result(name, expected, actual, passed):
    validation_results.append({
        "Validation_Name": name,
        "Expected": str(expected),
        "Actual": str(actual),
        "Status": "PASS" if passed else "FAIL"
    })
    

In [0]:
expected_count = expected_df.count()
actual_count = target_df.count()

add_validation_result(
    "Record Count",
    expected_count,
    actual_count,
    expected_count == actual_count
)

In [0]:
# Missing records
missing_df = expected_df.exceptAll(target_df)
missing_count = missing_df.count()

add_validation_result(
    "Missing Records",
    0,
    missing_count,
    missing_count == 0
)

# Extra records
extra_df = target_df.exceptAll(expected_df)
extra_count = extra_df.count()

add_validation_result(
    "Extra Records",
    0,
    extra_count,
    extra_count == 0
)


In [0]:
duplicate_df = target_df.groupBy(
    "order_id", "product_id"
).count().filter(col("count") > 1)

duplicate_count = duplicate_df.count()

display(duplicate_df)

add_validation_result(
    "Duplicate Validation",
    0,
    duplicate_count,
    duplicate_count == 0
)

In [0]:
from pyspark.sql.functions import col


from pyspark.sql.functions import col

joined_df = expected_df.alias("s").join(
    target_df.alias("t"),
    on=["order_id", "product_id"],
    how="inner"
)

col_validation = joined_df.filter(
    ~(col("s.customer_id").eqNullSafe(col("t.customer_id"))) |
    ~(col("s.customer_name").eqNullSafe(col("t.customer_name"))) |
    ~(col("s.city").eqNullSafe(col("t.city"))) |
    ~(col("s.state").eqNullSafe(col("t.state"))) |
    ~(col("s.order_date").eqNullSafe(col("t.order_date"))) |
    ~(col("s.quantity").eqNullSafe(col("t.quantity"))) |
    ~(col("s.amount").eqNullSafe(col("t.amount")))
)

mismatch_count = col_validation.count()

display(col_validation)

add_validation_result(
    "Column-Level Validation",
    0,
    mismatch_count,
    mismatch_count == 0
)


In [0]:
expected_columns = sorted(expected_df.columns)
target_columns = sorted(target_df.columns)

missing = []
for item in expected_columns:
    if item not in target_columns:
        missing.append(item)

extra = []
for item in target_columns:
    if item not in expected_columns:
        extra.append(item)

print("Missing columns:", missing)
print("Extra columns:", extra)

add_validation_result(
    "Missing Columns",
    "None",
    missing if missing else "None",
    len(missing) == 0
)

add_validation_result(
    "Extra Columns",
    "None",
    extra if extra else "None",
    len(extra) == 0
)


In [0]:
expected_schema = dict(expected_df.dtypes)
target_schema = dict(target_df.dtypes)

add_validation_result(
    "Data Type Validation",
    expected_schema,
    target_schema,
    expected_schema == target_schema
)
   

In [0]:

total_count = 0

for item in target_df.columns:
    null_count = target_df.filter(col(item).isNull()).count()
    total_count = total_count + null_count

    print(item, ":", null_count)

    add_validation_result(
        "Null Validation: " + item,
        0,
        null_count,
        null_count == 0
    )

print("Total null values across all columns:", total_count)


In [0]:
report_df = spark.createDataFrame(validation_results).select(
    "Validation_Name", "Expected", "Actual", "Status"
)

display(report_df)

failed_count = report_df.filter(
    col("Status") == "FAIL"
).count()

if failed_count > 0:
    raise RuntimeError(
        f"{failed_count} validation(s) failed. Check the reconciliation report."
    )

print("All reconciliation validations passed.")

In [0]:

# Save reconciliation report to Unity Catalog Volume

report_dir = (
    "/Volumes/reconciliation/default/reconciliation_files/"
    "reconciliation_report_output"
)

report_file_path = (
    "/Volumes/reconciliation/default/reconciliation_files/"
    "reconciliation_report.csv"
)

# Write the report as a CSV folder
report_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(report_dir)

# Find the generated CSV file
part_csv_path = None

for file_info in dbutils.fs.ls(report_dir):
    if file_info.name.startswith("part-") and file_info.name.endswith(".csv"):
        part_csv_path = file_info.path
        break

if part_csv_path is None:
    raise FileNotFoundError("CSV report was not generated.")

# Replace the previous report, if one exists
dbutils.fs.rm(report_file_path)

dbutils.fs.cp(part_csv_path, report_file_path)

print(f"Reconciliation report saved to: {report_file_path}")


In [0]:
# Negative Test:
# Intentionally modify the amount for product P102
# to verify that column-level reconciliation detects the mismatch.

from pyspark.sql.functions import col, when

bad_target_df = target_df.withColumn(
    "amount",
    when(col("product_id") == "P102", 999.0)
    .otherwise(col("amount"))
)

display(bad_target_df)

In [0]:
#negative test
from pyspark.sql.functions import col


from pyspark.sql.functions import col

joined_df = expected_df.alias("s").join(
    bad_target_df.alias("t"),
    on=["order_id", "product_id"],
    how="inner"
)

col_validation = joined_df.filter(
    ~(col("s.customer_id").eqNullSafe(col("t.customer_id"))) |
    ~(col("s.customer_name").eqNullSafe(col("t.customer_name"))) |
    ~(col("s.city").eqNullSafe(col("t.city"))) |
    ~(col("s.state").eqNullSafe(col("t.state"))) |
    ~(col("s.order_date").eqNullSafe(col("t.order_date"))) |
    ~(col("s.quantity").eqNullSafe(col("t.quantity"))) |
    ~(col("s.amount").eqNullSafe(col("t.amount")))
)

mismatch_count = col_validation.count()

display(col_validation)
print("Mismatched records:", mismatch_count)


